# 26 — Customer Onboarding Acceleration

**Why this matters at ServiceTitan**: there's a current open **Senior Data
Scientist (Onboarding)** req with this as the explicit team focus: "optimize
our customer onboarding process" and "improving the speed and quality of our
customer transitions." Onboarding is also where churn-prevention work pays off
the most — a contractor who doesn't reach first-value in 60 days is 4-5x more
likely to churn within the year.

This notebook covers the four primitives an onboarding DS team uses:

| Section | Technique | What it answers |
|---------|-----------|-----------------|
| 1 | Time-to-first-job survival analysis | "How long until a new contractor books their first job through us?" |
| 2 | Milestone propensity scoring | "Which contractors are likely to stall at this onboarding step?" |
| 3 | Bottleneck detection | "Which step in the funnel causes the most drop-off?" |
| 4 | GenAI personalized assistance | "What guidance should we send to *this* stuck contractor?" |
| 5 | Routing: ML vs GenAI vs human CS | "Who handles which intervention?" |
| 6 | Eval: did we reduce TTFJ? | Section 25's tools, applied |

All synthetic data, runs in seconds.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

np.random.seed(0)


---
## Section 1 — Time-to-First-Job: Survival Analysis

A new contractor signs up. The question for the onboarding team:

> How long does it take to book their first job through ServiceTitan?

This is a **survival analysis** problem. The unique difficulty: many contractors
are **right-censored** — they're still active and haven't yet booked. You can't
just compute `mean(days_to_first_job)` because you'd exclude exactly the
contractors you're worried about (the slow ones).

### Kaplan-Meier estimator (from scratch)

The KM survival function S(t) = P(no first job yet by day t):

$$
\hat{S}(t) = \prod_{t_i \le t} \left(1 - \frac{d_i}{n_i}\right)
$$

where at each observed event time t_i, d_i = events (first jobs) and n_i =
contractors still at-risk.


In [ ]:
# ── Generate synthetic cohort ──────────────────────────────────────────────
rng = np.random.default_rng(42)
N = 2000

# Each contractor has a "true" time-to-first-job drawn from a mixture:
# - 70% reach first job quickly (Weibull, scale=20 days)
# - 20% are slow (Weibull, scale=80 days)
# - 10% will never make it (effectively infinite)
fast = rng.weibull(1.5, N) * 20
slow = rng.weibull(1.5, N) * 80
never = np.full(N, 1e6)
class_probs = rng.choice([0, 1, 2], size=N, p=[0.7, 0.2, 0.1])
true_ttfj = np.where(class_probs == 0, fast, np.where(class_probs == 1, slow, never))

# Observation window: 90 days. After that, censor.
WINDOW = 90
observed_event = true_ttfj <= WINDOW
observed_time = np.minimum(true_ttfj, WINDOW)

df = pd.DataFrame({
    "ttfj_observed": observed_time,
    "had_first_job": observed_event,
})
print(f"Cohort: {N} contractors observed over {WINDOW} days")
print(f"  {observed_event.sum():,} had first job in window ({observed_event.mean():.1%})")
print(f"  {(~observed_event).sum():,} censored (no first job yet)")


In [ ]:
# ── Kaplan-Meier from scratch ─────────────────────────────────────────────
def kaplan_meier(times: np.ndarray, events: np.ndarray):
    """Return (sorted_times, survival_probs). Standard KM estimator."""
    order = np.argsort(times)
    t = times[order]
    e = events[order]

    unique_t = []
    s_vals = []
    s_current = 1.0
    n_at_risk = len(t)
    i = 0
    while i < len(t):
        j = i
        d = 0   # events at this time
        while j < len(t) and t[j] == t[i]:
            if e[j]:
                d += 1
            j += 1
        if d > 0:
            s_current *= (1 - d / n_at_risk)
        unique_t.append(t[i])
        s_vals.append(s_current)
        n_at_risk -= (j - i)
        i = j
    return np.array(unique_t), np.array(s_vals)


t_grid, s = kaplan_meier(df["ttfj_observed"].values, df["had_first_job"].values)

# Median survival = first time S(t) <= 0.5
median_idx = np.argmax(s <= 0.5) if (s <= 0.5).any() else None
median_t = t_grid[median_idx] if median_idx is not None else "not reached"
print(f"Median time-to-first-job: {median_t} days")
print(f"\n  S(30 days) = {s[t_grid <= 30][-1]:.3f}  ({(1-s[t_grid <= 30][-1]):.1%} had first job by day 30)")
print(f"  S(60 days) = {s[t_grid <= 60][-1]:.3f}  ({(1-s[t_grid <= 60][-1]):.1%} by day 60)")
print(f"  S(90 days) = {s[t_grid <= 90][-1]:.3f}  ({(1-s[t_grid <= 90][-1]):.1%} by day 90)")

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.step(t_grid, s, where="post")
ax.axhline(0.5, color="r", linestyle="--", alpha=0.5, label="median")
ax.set_xlabel("Days since signup")
ax.set_ylabel("P(no first job yet)")
ax.set_title("Kaplan-Meier survival curve: time to first job")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig("/tmp/km_curve.png", dpi=80)
plt.close()
print("\nSaved curve to /tmp/km_curve.png")


---
## Section 2 — Milestone Propensity Scoring

Survival analysis tells you the *population* curve. To intervene, you need a
*per-contractor* score: **how likely is this specific contractor to stall?**

We build a binary classifier: given features observable at signup + day-7
behavior, predict whether they'll hit "first job" by day 60.

### Features

- Static: company size, plan tier, industry
- Day-7 behaviors: did they complete setup wizard, upload customers, schedule first job

### Why GBM
Tabular data, mixed types, modest sample size, non-linear interactions matter.
Gradient boosting (XGBoost / LightGBM / sklearn) consistently wins this problem
class. We use sklearn's GBM for portability.


In [ ]:
# ── Synthetic feature dataset ─────────────────────────────────────────────
rng = np.random.default_rng(7)
N = 5000

shop_size = rng.choice([1, 2, 5, 10, 25], N, p=[0.4, 0.25, 0.20, 0.10, 0.05])
plan = rng.choice(["basic", "pro", "enterprise"], N, p=[0.6, 0.3, 0.1])
industry = rng.choice(["hvac", "plumbing", "electrical", "multi"], N)

# Day-7 engagement signals
setup_completed = rng.random(N) < (0.4 + 0.3 * (shop_size >= 5) + 0.2 * (plan != "basic"))
customers_uploaded = rng.binomial(50, 0.3 * setup_completed + 0.05, N)
first_job_scheduled_by_d7 = rng.random(N) < (0.2 + 0.4 * setup_completed)

# Target: first job by day 60
# Drivers: setup completed, customers uploaded, first job scheduled, plan, size
logit = (
    -1.5
    + 1.5 * setup_completed
    + 0.02 * customers_uploaded
    + 2.0 * first_job_scheduled_by_d7
    + 0.5 * (plan == "pro")
    + 0.8 * (plan == "enterprise")
    + 0.1 * (shop_size >= 5)
)
p_success = 1 / (1 + np.exp(-logit))
hit_first_job_by_60 = rng.random(N) < p_success

df = pd.DataFrame({
    "shop_size": shop_size,
    "plan_pro": (plan == "pro").astype(int),
    "plan_enterprise": (plan == "enterprise").astype(int),
    "industry_hvac": (industry == "hvac").astype(int),
    "industry_plumbing": (industry == "plumbing").astype(int),
    "industry_electrical": (industry == "electrical").astype(int),
    "setup_completed_d7": setup_completed.astype(int),
    "customers_uploaded_d7": customers_uploaded,
    "first_job_scheduled_d7": first_job_scheduled_by_d7.astype(int),
    "first_job_by_60": hit_first_job_by_60.astype(int),
})
print(f"Dataset: {len(df)} contractors")
print(f"  Base rate of first-job-by-60: {df['first_job_by_60'].mean():.1%}")


In [ ]:
# ── Train propensity model ────────────────────────────────────────────────
X = df.drop(columns="first_job_by_60")
y = df["first_job_by_60"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

model = GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=0)
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, proba)
print(f"Test AUC: {auc:.3f}")

# Feature importance
imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop drivers of success:")
for feat, val in imp.head(5).items():
    print(f"  {feat:<28} {val:.3f}")

# ── Risk segmentation: who needs intervention? ────────────────────────────
risk = 1 - proba  # high = likely to stall
high_risk = risk > 0.5
print(f"\nFlagged {high_risk.sum()} ({high_risk.mean():.1%}) test contractors as HIGH RISK of not reaching first job")
print(f"  Of those, {y_test[high_risk].sum()} actually didn't reach first job")
print(f"  → precision@high_risk: {1 - y_test[high_risk].mean():.1%}")


---
## Section 3 — Onboarding Funnel Bottleneck Detection

The propensity model identifies *who* is stuck. The funnel analysis identifies
*where* they get stuck.

Onboarding has discrete steps:
1. Account created
2. Setup wizard completed
3. Customers imported
4. Pricebook configured
5. First job scheduled
6. First job completed (= activation)

Compute per-step conversion rates, identify the worst step, and flag its
inputs as the bottleneck.


In [ ]:
# Synthetic funnel
steps = [
    ("account_created",        N),
    ("setup_wizard_done",      int(N * 0.78)),
    ("customers_imported",     int(N * 0.62)),
    ("pricebook_configured",   int(N * 0.31)),    # bottleneck!
    ("first_job_scheduled",    int(N * 0.27)),
    ("first_job_completed",    int(N * 0.24)),
]

print(f"{'Step':<28} {'N':>6} {'%':>7} {'step Δ%':>9}")
prev = N
for name, n in steps:
    pct_total = n / N * 100
    drop = (prev - n) / prev * 100 if prev else 0
    print(f"  {name:<28} {n:>6,} {pct_total:>6.1f}% {drop:>8.1f}%")
    prev = n

# Largest single-step drop
drops = [(steps[i][0], (steps[i-1][1] - steps[i][1]) / steps[i-1][1])
         for i in range(1, len(steps))]
worst = max(drops, key=lambda x: x[1])
print(f"\n🔴 BOTTLENECK: '{worst[0]}' (drop = {worst[1]:.1%})")
print(f"\nThis is where the onboarding team should focus:")
print(f"  - Could be UX (pricebook config is hard)")
print(f"  - Could be data prep (contractors don't have pricebook ready)")
print(f"  - Could be missing self-service guidance (no template, no examples)")


---
## Section 4 — GenAI Personalized Onboarding Assistance

Once you know **who** is stuck and **where**, generate the right intervention.
This is exactly where the Senior DS (Onboarding) JD points to: "Generative AI"
for "improving the efficiency of customer onboarding."

### Prompt design pattern

Templated system prompt with structured context:

```
You are an onboarding coach for ServiceTitan customers. The contractor below
has stalled at <step>. Their context is:

  - Plan: <plan>
  - Industry: <industry>
  - Shop size: <size>
  - Days since signup: <days>
  - Completed milestones: <list>

Generate a short, friendly message (under 100 words) suggesting their next
single action. Be specific to their context. End with a clear call-to-action.

DO NOT:
  - Mention competitors
  - Discuss pricing
  - Make promises about specific outcomes
```

We stub the LLM with deterministic templates so the notebook runs offline.


In [ ]:
# ── Stubbed LLM that returns templated messages ──────────────────────────
def generate_intervention(contractor: dict) -> str:
    """Stubbed LLM. Produces a personalized message based on the stuck step."""
    step = contractor["stuck_at"]
    plan = contractor["plan"]
    size = contractor["shop_size"]
    industry = contractor["industry"]
    days = contractor["days_since_signup"]

    templates = {
        "pricebook_configured": (
            f"Hi! It's been {days} days and we noticed you haven't configured your pricebook yet. "
            f"For a {industry} shop your size, the fastest path is our pre-built {industry.title()} "
            f"Pro Pricebook template (we'll auto-populate {'150+' if plan != 'basic' else '50'} "
            f"common services). Want me to load it in your account? Just reply YES."
        ),
        "customers_imported": (
            f"Welcome back! You're {days} days in and we can have your customer list live in under "
            f"5 minutes. Upload a CSV from your old system, or use our QuickBooks/Excel connector. "
            f"{'Your Pro plan includes white-glove import — want to book a 15-min session?' if plan == 'pro' else 'Need a template? Reply CSV.'}"
        ),
        "setup_wizard_done": (
            f"You're {days} days in and almost there! The setup wizard takes about 12 minutes — it "
            f"sets your business hours, service area, and tech roster. "
            f"{'Want a 15-min walkthrough call?' if size >= 5 else 'Reply START and we will send the link.'}"
        ),
        "first_job_scheduled": (
            f"Last step! Your pricebook is in, your customers are loaded. Schedule a first job to see "
            f"the full magic — dispatch, mobile app, invoicing all flow from here. Pick a customer "
            f"and add a service from your pricebook. We're here if you get stuck."
        ),
    }
    return templates.get(step, f"Welcome! It's been {days} days since signup. How can we help you get to your first job?")


# Example interventions
contractors_stuck = [
    {"stuck_at": "pricebook_configured", "plan": "pro", "shop_size": 5,  "industry": "hvac",     "days_since_signup": 14},
    {"stuck_at": "customers_imported",   "plan": "basic", "shop_size": 1, "industry": "plumbing", "days_since_signup": 7},
    {"stuck_at": "setup_wizard_done",    "plan": "pro", "shop_size": 10, "industry": "electrical","days_since_signup": 21},
]

for c in contractors_stuck:
    print(f"\n--- Contractor stuck at: {c['stuck_at']} ---")
    print(f"  Context: {c['plan']} plan, {c['shop_size']}-tech {c['industry']}, day {c['days_since_signup']}")
    print(f"  Message: {generate_intervention(c)}")


---
## Section 5 — Routing: ML vs GenAI vs Human

Not every stuck contractor needs the same intervention. The economics:

| Tier | Cost / touch | Best for |
|------|-------------|----------|
| **Automated email (template)** | $0.01 | Low-risk, common stall patterns |
| **GenAI personalized message** | $0.05 | Medium-risk, where personalization adds lift |
| **Human CS rep (15-min call)** | $50 | High-value (Enterprise plan, large shop) OR high-risk (likely to churn) |

### Routing rule (simple)

```
if (risk_score > 0.7 AND plan == "enterprise") OR shop_size > 25:
    → human CS rep
elif risk_score > 0.5:
    → GenAI personalized message
else:
    → automated email template
```

In practice the thresholds get tuned via the exact tools from notebook 25
(causal experiments on intervention type).


In [ ]:
def route_intervention(risk_score: float, plan: str, shop_size: int) -> str:
    """Decide which onboarding intervention to apply."""
    if risk_score > 0.7 and plan == "enterprise":
        return "human_cs"
    if shop_size > 25:
        return "human_cs"
    if risk_score > 0.5:
        return "genai_personalized"
    return "template_email"


# Apply to test set
test_df = X_test.copy()
test_df["risk"] = risk
test_df["plan"] = test_df["plan_pro"].map({1: "pro", 0: "basic"})
test_df.loc[test_df["plan_enterprise"] == 1, "plan"] = "enterprise"
test_df["route"] = test_df.apply(lambda r: route_intervention(r["risk"], r["plan"], r["shop_size"]), axis=1)

print("Routing distribution on test cohort:")
counts = test_df["route"].value_counts()
costs = {"template_email": 0.01, "genai_personalized": 0.05, "human_cs": 50}
total_cost = 0
for r, n in counts.items():
    c = costs[r] * n
    total_cost += c
    print(f"  {r:<22} {n:>4d}  ({n/len(test_df):.1%})   cost: ${c:>9,.2f}")
print(f"  {'TOTAL':<22} {len(test_df):>4d}             cost: ${total_cost:>9,.2f}")
print(f"  Cost per contractor: ${total_cost / len(test_df):.2f}")

# Compare to "always human CS" baseline
baseline_cost = len(test_df) * costs["human_cs"]
print(f"\n  vs. always-human:  ${baseline_cost:,.2f}  → {(1 - total_cost/baseline_cost):.0%} cost reduction")
print(f"  vs. always-email:  ${len(test_df) * costs['template_email']:,.2f}  → +${total_cost - len(test_df)*costs['template_email']:,.2f}")
print(f"  (the extra spend on GenAI/human goes to highest-risk segments where lift matters most)")


---
## Section 6 — Evaluation: Did We Actually Reduce TTFJ?

The success metric for onboarding is **time-to-first-job** (the survival
analysis target from Section 1). Use the methods from notebook 25.

### Why not just compare pre- and post-intervention cohorts?

Because:
- Seasonal effects (Q4 contractors onboard slower)
- Product changes (UI redesign in the middle)
- Selection (better contractors might have started joining in the post period)

The cleanest design is a **randomized A/B** on the intervention itself:
contractors flagged as high-risk are randomly assigned to template vs GenAI
vs human CS. Compare TTFJ across arms.

### Simulated experiment

We simulate this experiment and apply the analysis from notebook 25.


In [ ]:
# ── Simulate intervention A/B ─────────────────────────────────────────────
rng = np.random.default_rng(9)
n_per_arm = 800

# Arm 1: template email. Baseline.
ttfj_template = rng.gamma(shape=2, scale=20, size=n_per_arm)

# Arm 2: GenAI message. Modest lift.
ttfj_genai = rng.gamma(shape=2, scale=18, size=n_per_arm)

# Arm 3: human CS call. Bigger lift, more variance.
ttfj_human = rng.gamma(shape=2, scale=15, size=n_per_arm)

print("TTFJ by arm (high-risk segment only):")
for name, arm in [("template", ttfj_template), ("genai", ttfj_genai), ("human_cs", ttfj_human)]:
    print(f"  {name:<10}  mean={arm.mean():.1f}d  median={np.median(arm):.1f}d  p90={np.percentile(arm,90):.1f}d")

# Pairwise tests vs template
t_g, p_g = stats.ttest_ind(ttfj_genai, ttfj_template)
t_h, p_h = stats.ttest_ind(ttfj_human, ttfj_template)
print(f"\nGenAI vs template:    Δmean={ttfj_genai.mean()-ttfj_template.mean():+.1f}d  p={p_g:.4f}")
print(f"Human  vs template:    Δmean={ttfj_human.mean()-ttfj_template.mean():+.1f}d  p={p_h:.4f}")

# Cost-effectiveness (days saved per dollar)
days_saved_per_dollar_genai = (ttfj_template.mean() - ttfj_genai.mean()) / (costs["genai_personalized"] - costs["template_email"])
days_saved_per_dollar_human = (ttfj_template.mean() - ttfj_human.mean()) / (costs["human_cs"] - costs["template_email"])
print(f"\nDays saved per incremental $:")
print(f"  GenAI:    {days_saved_per_dollar_genai:.1f} days/$")
print(f"  Human CS: {days_saved_per_dollar_human:.3f} days/$")
print(f"\n  GenAI is hugely more cost-effective per dollar — but human CS reaches contractors")
print(f"  GenAI can't (the high-value enterprise accounts where the absolute days saved matter most).")


---
## End-to-End Pipeline Summary

```
Signup → [day 7 features collected]
                ↓
      [Section 2: propensity model]
                ↓
       risk_score per contractor
                ↓
      [Section 5: routing decision]
                ↓
   ┌────────────┼────────────┐
   ↓            ↓            ↓
template      GenAI        human CS
                ↓
       [Section 6: eval]
                ↓
      iterate weights & thresholds
```

This is the loop the Senior DS (Onboarding) role would run quarterly:
1. Train the propensity model on the latest cohort
2. Re-tune routing thresholds based on previous quarter's A/B results
3. Spot-check the GenAI prompt for hallucinations or drift (notebook 22's
   guardrails)
4. Re-segment if a new persona emerges (e.g., roofing contractors after
   ST acquires a roofing-specialty business)


In [ ]:
print("=" * 60)
print("Notebook 26 — Customer Onboarding Acceleration")
print("=" * 60)
print(f"  Section 1 — survival analysis (KM):    median TTFJ computed")
print(f"  Section 2 — propensity model:          AUC = {auc:.3f}")
print(f"  Section 3 — bottleneck detection:      identified worst funnel step")
print(f"  Section 4 — GenAI personalization:     3 sample interventions generated")
print(f"  Section 5 — routing decision:          3-tier cost-aware routing")
print(f"  Section 6 — A/B evaluation:            template vs genai vs human compared")
print("=" * 60)
